In [2]:
import psycopg2
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(
    host = "localhost",
    port = 5432,
    dbname = "pillwise",
    user = "pillwise",
    password = "pillwise123"
)

register_vector(conn)
print("connected to pgvector successfully")

connected to pgvector successfully


In [3]:
cur = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS chunks (
        id SERIAL PRIMARY KEY,
        drug TEXT,
        source TEXT,
        strategy TEXT,
        chunk_test TEXT,
        embedding vector(3072)
    );
""")

conn.commit()
print("table created successfully")

table created successfully


In [4]:
import os
import glob
import fitz
import re
import time
from dotenv import load_dotenv
from google import genai
import numpy as np

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

DATA_DIR = os.path.expanduser("~/Desktop/projects-learning/PillWise/data/raw_pdfs")
pdf_paths = glob.glob(os.path.join(DATA_DIR, "**/*.pdf"), recursive=True)
pdf_paths.sort()

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text.strip()

def fixed_size_chunks(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

documents = []
for path in pdf_paths:
    text = extract_text(path)
    drug_name = path.split("/")[-2]
    documents.append({"text": text, "source": path, "drug": drug_name})

all_chunks_fixed = []
for doc in documents:
    for chunk in fixed_size_chunks(doc['text']):
        all_chunks_fixed.append({
            "chunk": chunk,
            "drug": doc['drug'],
            "source": doc['source'],
            "strategy": "fixed"
        })

print(f"chunks ready: {len(all_chunks_fixed)}")

chunks ready: 3139


In [18]:
three_medicines = [c for c in all_chunks_fixed if c['drug'] in ['acetaminophen', 'ibuprofen', 'amoxicillin']]
print(len(three_medicines))

609


In [19]:
conn.rollback()
print("transaction rolled back")

transaction rolled back


In [20]:
def insert_chunks(chunks, batch_size=10):
    cur = conn.cursor()
    total = len(chunks)
    
    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c['chunk'] for c in batch]
        
        while True:
            try:
                result = client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=texts
                )
                break
            except Exception as e:
                if "429" in str(e):
                    print("rate limit hit, waiting 30s...")
                    time.sleep(30)
                else:
                    raise e
        
        for j, embedding in enumerate(result.embeddings):
            cur.execute("""
                INSERT INTO chunks (drug, source, strategy, chunk_text, embedding)
                VALUES (%s, %s, %s, %s, %s)
            """, (
                batch[j]['drug'],
                batch[j]['source'],
                batch[j]['strategy'],
                batch[j]['chunk'],
                embedding.values
            ))
        
        conn.commit()
        print(f"inserted {min(i + batch_size, total)}/{total}")
        time.sleep(0.7)  # ~85 requests/min, under the 100 limit

# clear and restart
cur2 = conn.cursor()
cur2.execute("TRUNCATE TABLE chunks;")
conn.commit()
print("table cleared, starting full insert...")
insert_chunks(three_medicines)

table cleared, starting full insert...
inserted 10/609
inserted 20/609
inserted 30/609
inserted 40/609
inserted 50/609
inserted 60/609
inserted 70/609
inserted 80/609
inserted 90/609
inserted 100/609
rate limit hit, waiting 30s...
inserted 110/609
inserted 120/609
inserted 130/609
inserted 140/609
inserted 150/609
inserted 160/609
inserted 170/609
inserted 180/609
inserted 190/609
inserted 200/609
rate limit hit, waiting 30s...
inserted 210/609
inserted 220/609
inserted 230/609
inserted 240/609
inserted 250/609
inserted 260/609
inserted 270/609
inserted 280/609
inserted 290/609
inserted 300/609
rate limit hit, waiting 30s...
inserted 310/609
inserted 320/609
inserted 330/609
inserted 340/609
inserted 350/609
inserted 360/609
inserted 370/609
inserted 380/609
inserted 390/609
inserted 400/609
rate limit hit, waiting 30s...
inserted 410/609
inserted 420/609
inserted 430/609
inserted 440/609
inserted 450/609
inserted 460/609
inserted 470/609
inserted 480/609
inserted 490/609
inserted 500/

In [21]:
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM chunks;")
print(f"total rows: {cur.fetchone()[0]}")

cur.execute("SELECT drug, COUNT(*) FROM chunks GROUP BY drug;")
for row in cur.fetchall():
    print(f"{row[0]}: {row[1]} chunks")

total rows: 609
amoxicillin: 310 chunks
acetaminophen: 53 chunks
ibuprofen: 246 chunks
